Load Packages

In [2]:
import pandas as pd
import json
from pathlib import Path
from datetime import datetime


Load data from directory and analyse 

In [3]:
file_path = Path("Data/raw/preprocessed_capstone2025.json")
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.json_normalize(data, sep="_")  # Flattens nested structure


FileNotFoundError: [Errno 2] No such file or directory: 'Data\\raw\\preprocessed_capstone2025.json'

In [ ]:
# Flatten the JSON into a structured DataFrame

records = []
for doc_id, content in data.items():
    base = {
        "doc_id": doc_id.removesuffix(".xml"),
        "schema": content.get("schema"),
        "dokument_art": content.get("dokument_art", [None])[0],
        "dokument_typ": content.get("dokument_typ", [None])[0],
        "institution": content.get("bibliographische-angaben", {}).get("institution", [None])[0],
        "aktenzeichen": content.get("bibliographische-angaben", {}).get("aktenzeichen", [None])[0],
        "datum": pd.to_datetime(content.get("bibliographische-angaben", {}).get("datum", [None])[0], format="%Y-%m-%d", errors="coerce"),
        "fundstelle": "; ".join(content.get("bibliographische-angaben", {}).get("fundstelle_kuerzel", [])),
        "vorinstanz": "; ".join(content.get("bibliographische-angaben", {}).get("vorinstanz", [])),
        "norm_vorinstanz_aktenzeichen": "; ".join(content.get("bibliographische-angaben", {}).get("norm_vorinstanz_aktenzeichen", [])),
        "norm_kuerzel": "; ".join(content.get("bibliographische-angaben", {}).get("norm_kuerzel", [])),
        "titel": content.get("text", {}).get("titel", [None])[0],
        "leitsatz": "; ".join(content.get("text", {}).get("entscheidungsinhalt", {}).get("leitsatz", [])),        
        "gruende": "; ".join([v for k, v in sorted(content.get("text", {}).get("gruende", {}).get("gruende", {}).items(), key=lambda x: int(x[0]))]),
        "rthema": "; ".join(content.get("allgemeine-angaben", {}).get("rthema", [])),
        "quelle": content.get("interne-angaben", {}).get("quelle", [None])[0]
    }
    records.append(base)

# Create a DataFrame
df = pd.DataFrame(records)


Handle misslabled rthemen

In [ ]:
# Define replacements for malformed themes
rthema_corrections = {
    "HandelsrechtGesellschaftsrecht": "Handelsrecht Gesellschaftsrecht",
    "MietePacht": "Miete Pacht"
}

def correct_rthema(rthema_str):
    if pd.isna(rthema_str):
        return rthema_str
    for wrong, correct in rthema_corrections.items():
        rthema_str = rthema_str.replace(wrong, correct)
    return rthema_str

df["rthema"] = df["rthema"].apply(correct_rthema)


Remove documents we do not want to use

In [ ]:
allowed_rthemen = {"Zivilverfahrensrecht", "SchuldrechtAT", "Schadensersatz", "Handelsrecht Gesellschaftsrecht", "BGBAT", "Miete Pacht", "Sachenrecht", "Versicherungsrecht","Kauf Tausch Leasing","Erbschaft Schenkung", "Schuldverhältnisse", "Werkvertrag", "EU-Recht", "Wohnungseigentum", "Sonstiges Recht", "Reisevertrag"}
def rthema_valid(rthema_string, allowed_set):
    if pd.isna(rthema_string):
        return False
    themen = set(rthema_string.split("; "))
    return themen.issubset(allowed_set)

df_filtered = df[
    df["dokument_art"].isin(["Urteil"]) &
    df["dokument_typ"].isin(["Obere Rechtsprechung"]) &
    df["institution"].notna() &
    (df["datum"] > pd.Timestamp("2000-01-01")) &
    df["rthema"].apply(lambda x: rthema_valid(x, allowed_rthemen))
].reset_index(drop=True)


In [ ]:
print(len(df_filtered))

Save csv

In [ ]:
df_filtered.to_csv("Data/clean/filtered_fabi.csv", index=False)